# Break-My-VLA · M4 Auto-Chaos

在 Colab 选择 G4 或 L4，再点 **Run all**。本地 Linux 或 WSL 上的 RTX 5090 可以从同一分支运行。默认评测 SmolVLA、Pi0.5 和 Isaac GR00T，覆盖 BF16、FP8、INT8、NVFP4 与 0–5 级。L4 会将 NVFP4 记录为跳过。

先用 Hugging Face 账号获取 PaliGemma 与 Cosmos Reason2 的仓库访问权限，再将有权访问的 `HF_TOKEN` 加到 Colab Secrets 并允许本 Notebook 读取。本地使用已登录的 Hugging Face 凭据。[PaliGemma](https://huggingface.co/google/paligemma-3b-pt-224) · [Cosmos Reason2](https://huggingface.co/nvidia/Cosmos-Reason2-2B)

Colab 将每次运行的指标、GIF、校准样本与导出产物写入 `MyDrive/Break-My-VLA/results/m4_auto_chaos/`。本地默认写入 `benchmarks/results/m4_auto_chaos/`。

## 配置评测范围

默认每档运行完整等级矩阵。快速检查可将 `BMVLA_LEVELS` 设为 `0,5`，或设置 `BMVLA_MOCK=1` 用 CPU 跑相同的导出、评测和报表入口。

In [ ]:
import json
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZipFile

MOCK = os.environ.get("BMVLA_MOCK") == "1"
MODELS = os.environ.get("BMVLA_MODELS", "smolvla,pi05,groot")
PRECISIONS = os.environ.get("BMVLA_PRECISIONS", "bf16,fp8,int8,nvfp4")
TASKS = os.environ.get("BMVLA_TASKS", "libero_object:0")
LEVELS = os.environ.get("BMVLA_LEVELS", "0-5")
SEED = int(os.environ.get("BMVLA_SEED", "0"))
MAX_STEPS = int(os.environ.get("BMVLA_MAX_STEPS", "40" if MOCK else "280"))
GROOT_SUITE = os.environ.get("BMVLA_GROOT_SUITE")

## 定位源码和结果目录

可以用 `BMVLA_REPO_DIR` 指定现有分支目录。Colab 也可从 Drive 的 `Break-My-VLA/Break-My-VLA-m4.zip` 解压源码，或直接克隆 M4 分支。

In [ ]:
try:
    from google.colab import drive, userdata
except ImportError:
    drive = userdata = None

drive_dir = None
if drive is not None and not MOCK:
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/Break-My-VLA")
    model_set = {name.strip() for name in MODELS.split(",")}
    if model_set & {"pi05", "groot"} and not os.environ.get("HF_TOKEN"):
        try:
            token = userdata.get("HF_TOKEN")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            token = None
        if token:
            os.environ["HF_TOKEN"] = token

marker = Path("scripts/m4_gpu_runtime.py")
if repo_dir := os.environ.get("BMVLA_REPO_DIR"):
    repo = Path(repo_dir).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    repo = next((path for path in (cwd, *cwd.parents) if (path / marker).is_file()), None)
    if repo is None and drive is not None:
        repo = Path("/content/Break-My-VLA-m4")
        archive = drive_dir / "Break-My-VLA-m4.zip" if drive_dir else None
        if not repo.exists():
            if archive is not None and archive.is_file():
                with ZipFile(archive) as bundle:
                    bundle.extractall(repo)
            else:
                subprocess.run(
                    ["git", "clone", "--depth", "1", "--branch", "feat/m4-groot-trt",
                     "https://github.com/JasperXzy/Break-My-VLA.git", str(repo)],
                    check=True,
                )
if repo is None or not (repo / marker).is_file():
    raise FileNotFoundError("Set BMVLA_REPO_DIR to a checkout of feat/m4-groot-trt")

output_root = (
    drive_dir / "results/m4_auto_chaos"
    if drive_dir else repo / "benchmarks/results/m4_auto_chaos"
)
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
output = Path(os.environ.get("BMVLA_OUTPUT_DIR", str(output_root / stamp))).expanduser().resolve()
print(f"Results will be saved in {output}")

## 导出 TensorRT 并运行 Auto-Chaos

启动脚本通过 `uv` 创建独立的 Python 3.11 环境，使用 `--index-strategy unsafe-best-match` 与 `MPLBACKEND=Agg`。GPU 路径从不同 LIBERO 初始状态采集校准与测试输入，并逐个导出和评测模型。每个策略进程退出后释放显存。

BF16、FP8 和 NVFP4 的引擎覆盖 GR00T DiT 以及 SmolVLA、Pi0.5 的去噪输出投影。INT8 使用 PyTorch 权重量化。GPU 构建时间和磁盘占用取决于模型与硬件。

In [ ]:
command = [
    sys.executable, "-m", "scripts.m4_gpu_runtime", "--models", MODELS,
    "--precisions", PRECISIONS, "--tasks", TASKS, "--levels", LEVELS,
    "--max-steps", str(MAX_STEPS), "--seed", str(SEED), "--output", str(output),
]
if GROOT_SUITE:
    command.extend(("--groot-suite", GROOT_SUITE))
if runtime_dir := os.environ.get("BMVLA_RUNTIME_DIR"):
    command.extend(("--runtime-dir", runtime_dir))
elif drive is not None:
    command.extend(("--runtime-dir", "/content/bmvla-m4-runtime"))
if MOCK:
    command.append("--mock")
elif drive is not None:
    command.append("--install-osmesa")
subprocess.run(command, cwd=repo, check=True)

## 检查延迟、对齐结果与回放

榜单延迟来自 Clean 策略调用。Auto-Chaos 的动作误差基于各服务提供的 BF16 档位。每个模型的 `tensorrt/<model>/profile.json` 保存 TensorRT 组件的 CUDA Event 延迟和相对 PyTorch BF16 的端到端动作误差。[评测方法](../benchmarks/auto_chaos.md)

In [ ]:
from IPython.display import Image, Markdown, display

gallery = json.loads((output / "gallery.json").read_text(encoding="utf-8"))
metrics = json.loads((output / "metrics.json").read_text(encoding="utf-8"))

def display_value(value, pattern=".2f"):
    return format(value, pattern) if value is not None else "—"

table = [
    "| 模型 | 精度 | 后端 | 成功率得分 | Clean p99 (ms) | 动作 MSE vs BF16 |",
    "| --- | --- | --- | ---: | ---: | ---: |",
]
for entry in gallery["entries"]:
    latency = display_value(entry["latency"].get("p99_ms"))
    mse = display_value(entry.get("alignment", {}).get("action_mse_vs_bf16"), ".3g")
    model = entry["model"].split("/")[-1] if MOCK else entry["family"]
    table.append(
        f"| {model} | {entry['precision'].upper()} | {entry.get('backend', '—')} | "
        f"{entry['overall_score']:.1%} | {latency} | {mse} |"
    )
display(Markdown(f"**{gallery['suite_name']}** · {gallery['hardware']}"))
display(Markdown("\n".join(table)))

for model, policy in metrics["models"].items():
    if policy["skipped_precisions"]:
        print(f"{model} · skipped: {', '.join(policy['skipped_precisions'])}")
    profile = output / "tensorrt" / model / "profile.json"
    if profile.is_file():
        print(f"{model} · {profile.relative_to(output)}")

if gallery["replays"]:
    replay = gallery["replays"][0]
    display(Markdown(f"**{replay['title']}**"))
    for label in ("baseline", "degraded"):
        display(Markdown(replay[f"{label}_label"]))
        display(Image(filename=str(output / replay[f"{label}_gif"]), width=320))